# 03 GOLD
## Silver → business summary

Gold goal:

```text
Turn detailed Silver rows
into a simple business-facing result
```

Input:

```text
demo_orders_silver
126 rows
```

Output:

```text
demo_order_status_gold
4 rows
```

Gold grain:

```text
one row = one order status
```


In [0]:
# ============================================================
# SETUP — RESOLVE CATALOG / SCHEMAS / OBJECT NAMES
# ============================================================
# WHAT:
#   Rebuild the same names used by the previous notebook.
#
# WHY:
#   Variables do not automatically persist between separate notebooks.
#
# EXPECTED:
#   All layer object names resolve consistently.

ctx = spark.sql("""
SELECT
    current_catalog() AS catalog,
    current_schema() AS schema
""").first()

CATALOG = ctx["catalog"]
CURRENT_SCHEMA = ctx["schema"]

if CATALOG == "hive_metastore":
    raise RuntimeError(
        "Switch to a Unity Catalog catalog first."
    )

def qi(value: str) -> str:
    return f"`{value.replace('`', '``')}`"

available = {
    r["databaseName"]
    for r in spark.sql(
        f"SHOW SCHEMAS IN {qi(CATALOG)}"
    ).collect()
}

BRONZE_SCHEMA = (
    "retail_bronze"
    if "retail_bronze" in available
    else CURRENT_SCHEMA
)

SILVER_SCHEMA = (
    "retail_silver"
    if "retail_silver" in available
    else CURRENT_SCHEMA
)

GOLD_SCHEMA = (
    "retail_gold"
    if "retail_gold" in available
    else CURRENT_SCHEMA
)

SOURCE_FILE = (
    f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/"
    f"demo_source_files/orders/orders.csv"
)

BRONZE_TABLE = (
    f"{qi(CATALOG)}.{qi(BRONZE_SCHEMA)}."
    f"{qi('demo_orders_bronze')}"
)

SILVER_TABLE = (
    f"{qi(CATALOG)}.{qi(SILVER_SCHEMA)}."
    f"{qi('demo_orders_silver')}"
)

GOLD_TABLE = (
    f"{qi(CATALOG)}.{qi(GOLD_SCHEMA)}."
    f"{qi('demo_order_status_gold')}"
)

print("Source :", SOURCE_FILE)
print("Bronze :", BRONZE_TABLE)
print("Silver :", SILVER_TABLE)
print("Gold   :", GOLD_TABLE)


In [0]:
# ============================================================
# CELL 2 — READ SILVER
# ============================================================
# WHAT:
#   Read the cleaned Silver orders table.
#
# WHY:
#   Gold must depend on Silver, not directly on Bronze or source files.
#
# EXPECTED:
#   126 Silver rows.

from pyspark.sql import functions as F

silver_df = spark.table(
    SILVER_TABLE
)

assert silver_df.count() == 126


In [0]:
# ============================================================
# CELL 3 — BUILD THE GOLD BUSINESS SUMMARY
# ============================================================
# WHAT:
#   Aggregate orders by order_status.
#
# WHY:
#   Gold should answer a business question directly.
#
# INPUT GRAIN:
#   One row = one order.
#
# OUTPUT GRAIN:
#   One row = one order status.
#
# EXPECTED:
#   4 rows with order counts.

gold_df = (
    silver_df
    .groupBy(
        "order_status"
    )
    .agg(
        F.count(
            "*"
        ).alias(
            "order_count"
        ),
        F.min(
            "order_ts"
        ).alias(
            "first_order_ts"
        ),
        F.max(
            "order_ts"
        ).alias(
            "latest_order_ts"
        ),
    )
)

display(
    gold_df.orderBy(
        F.col("order_count").desc()
    )
)

assert gold_df.count() == 4


In [0]:
# ============================================================
# CELL 4 — WRITE THE GOLD DELTA TABLE
# ============================================================
# WHAT:
#   Persist the business summary as a managed Delta table.
#
# WHY:
#   BI/reporting users can query Gold directly without rebuilding logic.
#
# EXPECTED:
#   demo_order_status_gold contains 4 rows.

(
    gold_df.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        GOLD_TABLE
    )
)

gold_count = spark.table(
    GOLD_TABLE
).count()

print("Gold rows:", gold_count)

assert gold_count == 4


In [0]:
# ============================================================
# CELL 5 — FINAL PIPELINE VALIDATION
# ============================================================
# WHAT:
#   Validate the full Source → Bronze → Silver → Gold pipeline.
#
# WHY:
#   A demo should end with one clear reconciliation.
#
# EXPECTED:
#   Source=126, Bronze=126, Silver=126, Gold=4
#   and official status counts match.

bronze_count = spark.table(
    BRONZE_TABLE
).count()

silver_count = spark.table(
    SILVER_TABLE
).count()

gold_result = {
    row["order_status"]: row["order_count"]
    for row in spark.table(
        GOLD_TABLE
    ).collect()
}

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)
print("Gold rows  :", len(gold_result))
print("Gold result:", gold_result)

assert bronze_count == 126
assert silver_count == 126
assert gold_result == {
    "DELIVERED": 78,
    "SHIPPED": 29,
    "PLACED": 10,
    "CANCELLED": 9,
}

print(
    "\nSOURCE → BRONZE → SILVER → GOLD DEMO: PASSED"
)


# Final demo explanation

Use this in class:

```text
SOURCE
orders.csv
126 rows
   ↓
BRONZE
demo_orders_bronze
126 rows
raw-ish + ingestion metadata
   ↓
SILVER
demo_orders_silver
126 rows
cleaned + typed + standardized
   ↓
GOLD
demo_order_status_gold
4 rows
business summary
```

### One-sentence layer explanation

```text
Bronze
= what arrived

Silver
= cleaned and trusted detail

Gold
= business-ready summary
```
